# VayuNetra Evaluation Harness (Agent 0 & Agent 3)

This notebook evaluates the core multi-agent pipeline:
1. **Latency**: End-to-end signal-to-action time (Target: < 5 min).
2. **Orchestrator Routing Accuracy**: Does it correctly trigger enforcement on spikes?
3. **Enforcement Priority Correlation**: Do prioritized sources correlate with highest population exposure & pollution contribution?
4. **RAG Citation Relevance**: Precision of retrieved regulatory text against source type.

Run this in `DEMO_MODE=true` to test against fixtures, or `DEMO_MODE=false` for live DB traces.

In [ ]:
import os
import time
import json
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

# Force DEMO_MODE for local testing
os.environ["DEMO_MODE"] = "true"

from agents.graph import run_query
from agents.enforcement import run_enforcement
from rag.retrieve import retrieve_for_enforcement

## 1. Latency & Routing Evaluation
We run the LangGraph orchestrator 10 times and measure end-to-end latency.

In [ ]:
latencies = []
traces = []

print("Running orchestrator 10 times...")
for i in range(10):
    t0 = time.time()
    # Provide focus_cells to guarantee the spike-gate routes to Enforcement
    state = run_query("delhi", focus_cells=["883da1a3a1fffff"])
    latencies.append(state.get("latency_ms", int((time.time() - t0)*1000)))
    traces.append([t["node"] for t in state["trace"]])
    
df_lat = pd.DataFrame({"run": range(1, 11), "latency_ms": latencies})
print(f"\nAverage Latency: {df_lat['latency_ms'].mean():.2f} ms")
print(f"Max Latency: {df_lat['latency_ms'].max()} ms")
print("\nRouting Traces (first 3):")
for t in traces[:3]:
    print(" -> ".join(t))

## 2. Enforcement Priority Distribution
We evaluate how Agent 3 scores recommendations. We expect a healthy spread of scores.

In [ ]:
recs = run_enforcement("delhi")
df_recs = pd.DataFrame([r.to_dict() for r in recs])

print("Enforcement Priorities Generated:", len(df_recs))
display(df_recs[["source_id", "priority_score", "contribution", "pop_exposed", "status"]].head())

if not df_recs.empty:
    plt.figure(figsize=(8, 4))
    sns.scatterplot(data=df_recs, x="contribution", y="priority_score", size="pop_exposed", legend=False, alpha=0.7)
    plt.title("Priority Score vs. PM2.5 Contribution (Bubble size = Pop Exposed)")
    plt.xlabel("PM2.5 Contribution (Share)")
    plt.ylabel("Priority Score")
    plt.grid(True, alpha=0.3)
    plt.show()

## 3. RAG Retrieval Relevance
We check if `retrieve_for_enforcement` fetches the correct CPCB/GRAP rules.

In [ ]:
categories = ["construction_dust", "industrial", "biomass_burning"]
results = []

for cat in categories:
    chunks = retrieve_for_enforcement(cat, top_k=2)
    for c in chunks:
        results.append({
            "category": cat,
            "rule_title": c.title,
            "similarity": c.similarity,
        })
        
df_rag = pd.DataFrame(results)
display(df_rag)

## 5. Forecast Skill (Agent 2 — Omkar)

Walk-forward (3-fold) cross-validated skill on **live** PM2.5 data:
`skill = 1 - RMSE_model / RMSE_baseline` vs **persistence** and **climatology**.
Honest finding: persistence is a strong PM2.5 baseline at 24-48h; the model clearly
beats it at 72h and beats climatology — but the >=0.25-vs-persistence target is not met
under rigorous CV (the GNN/TFT upgrade is the Stage-2 attempt).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from core.supa import load_measurements
from ml.forecast.features import build_feature_table
from ml.forecast.train import backtest

wide = build_feature_table(pd.DataFrame(load_measurements("delhi")))
df_skill = pd.DataFrame([backtest(wide, h) for h in (24, 48, 72)])
display(df_skill[["horizon_h", "n", "rmse_model", "skill_vs_persistence", "skill_vs_climatology"]])

ax = df_skill.set_index("horizon_h")[["skill_vs_persistence", "skill_vs_climatology"]].plot(
    kind="bar", figsize=(7, 4))
ax.axhline(0.25, ls="--", color="red", label="target 0.25")
ax.axhline(0, color="black", lw=0.6)
plt.title("Agent 2 Forecast skill (walk-forward 3-fold CV)")
plt.ylabel("skill = 1 - RMSE_model / RMSE_baseline")
plt.legend(); plt.tight_layout(); plt.show()

## 6. Attribution / Blame Map (Agent 1 — Omkar)

The chemical-signature + satellite-fused blame map. We show the dominant-source spread
across cells and the city-wide mean share per source, plus mean confidence.
(Calibration against held-out SAFAR/TERI apportionment is the Stage-2 upgrade.)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from core.supa import client

rows = (client().table("attribution")
        .select("h3_cell,source_category,share,confidence")
        .eq("city_id", "delhi").execute().data)
adf = pd.DataFrame(rows)
dom = adf.loc[adf.groupby("h3_cell")["share"].idxmax()]
print("Dominant source per cell:")
display(dom["source_category"].value_counts())
print("Mean attribution confidence:", round(adf["confidence"].mean(), 3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
dom["source_category"].value_counts().plot(kind="bar", ax=axes[0], title="Dominant source per cell")
adf.groupby("source_category")["share"].mean().sort_values().plot(
    kind="barh", ax=axes[1], title="Mean source share (city-wide)")
plt.tight_layout(); plt.show()

## 7. Deep-forecast upgrade evaluation (Stage-2 §3A) — verdict: KEEP LightGBM

Per PLAN §3A (*"GNN/TFT upgrade — adopt only if it beats the baseline"*), an attention-augmented
quantile sequence model (TFT-lite: 2-layer LSTM + multi-head attention + pinball loss) was trained
on the live Delhi measurements (121,558 rows, 11 cells) on a Colab T4 GPU and evaluated with the
**identical walk-forward protocol** as the production model — see `notebooks/colab_tft_forecast.ipynb`.

Run: 2026-07-06 · skill = 1 − RMSE_model / RMSE_persistence

| horizon | LightGBM (prod) | TFT-lite | winner |
|---|---|---|---|
| 24h | **+0.036** | −0.250 | LightGBM |
| 48h | **+0.039** | −0.128 | LightGBM |
| 72h | **+0.078** | −0.079 | LightGBM |

**Verdict: production stays on `lgbm-q-v1`.** The deep model learns the diurnal cycle
(positive vs climatology at 24h: +0.053) but not the hour-to-hour residual that persistence
already captures — the expected outcome at this data scale (~27k sequences). The upgrade path
is more data (Stage-2 E2 dense coverage), not more architecture. Re-run the notebook to reproduce.


## 8. Attribution v2 — validity safeguards + physical plausibility (2026-07-06)

The hybrid GBM+SHAP apportionment (`ml/attribution/shap_attribution.py`) carries three safeguards:
1. **Observed-marker coverage mask** — a source earns SHAP blame only where its marker is actually
   measured (≥30% of the window); LightGBM's missing-value branch otherwise assigns identical
   phantom blame to every sensor-less cell.
2. **Holdout-R² gate (≥0.15)** — a model with no out-of-sample skill must not assign ML blame;
   such cities keep the transparent chemical-signature priors (Bengaluru currently falls back: R²=0.0).
3. **Prior blend + calibrated confidence** — 60/40 blend with chemistry priors; confidence from
   cross-method agreement + holdout R² + sample depth.

**Physical plausibility check (Delhi, R²=0.71):** mean positive traffic-marker SHAP is
**2.30× higher during IST rush hours** (08–10, 18–21) than off-peak (5.53 vs 2.41 µg/m³),
with boundary-layer height and wind controlled as model features — the ML blame follows real
traffic behaviour, not just diurnal meteorology.

*Known limitations (stated, not hidden):* SHAP measures model reliance, not emission inventory —
canonical PMF/CMB receptor models require chemically speciated data unavailable in India's public
feeds; `pm10_pm25_ratio` softly encodes the target (PM10 itself is excluded from features).


## 9. Prediction-interval calibration — found under-coverage, fixed with CQR (2026-07-08)

We audited the served [q0.1, q0.9] bands with a walk-forward coverage backtest
(`python -m ml.forecast.train --city <c> --coverage`). Raw quantile-LightGBM bands
covered only **48–63%** of outcomes against a nominal 80% — so we applied
**Conformalized Quantile Regression** (Romano et al. 2019): band width is calibrated on a
held-out split so the served intervals restore near-nominal coverage.

| City | raw coverage (24/48/72h) | CQR coverage | nominal |
|---|---|---|---|
| Delhi | 54% / 54% / 56% | **75% / 77% / 76%** | 80% |
| Bengaluru | 49% / 47% / 47% | **75% / 75% / 75%** | 80% |
| Mumbai | 59% / 50% / 52% | **80% / 76% / 71%** | 80% |

The residual gap to 80% is honest temporal distribution shift (calibration on the past,
evaluation on the future). Production now serves CQR-adjusted bands (`write_forecasts`).

*Registry hygiene (same audit):* water/sewage infrastructure is excluded from the OSM
emission registry (water polluters, not PM sources); enforcement matches each source to its
**own nearest attribution cell** instead of the city-dominant share, and recs below 2%
contribution are never emitted.
